In [53]:
import time
import numpy as np
from numpy import mean, std, dstack
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
from mlxtend.plotting import plot_confusion_matrix
from sklearn.metrics import confusion_matrix, matthews_corrcoef
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling1D, Dropout, InputLayer
from tensorflow.keras.callbacks import History, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Layer
import xlsxwriter
from tensorflow.keras.regularizers import l2
from keras import layers
import keras
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import MaxPooling1D
from tensorflow.keras.layers import LSTM, Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, Add, GlobalAveragePooling1D, Lambda
from tensorflow.keras.models import Model
import tensorflow as tf








In [54]:
class Time2Vec(Layer):
    def __init__(self, kernel_size=1):
        super(Time2Vec, self).__init__()
        self.k = kernel_size

    def build(self, input_shape):
        self.w0 = self.add_weight(name="w0", shape=(1,), initializer="uniform", trainable=True)
        self.b0 = self.add_weight(name="b0", shape=(1,), initializer="uniform", trainable=True)
        self.w = self.add_weight(name="w", shape=(input_shape[-1], self.k), initializer="uniform", trainable=True)
        self.b = self.add_weight(name="b", shape=(self.k,), initializer="uniform", trainable=True)

    def call(self, inputs):
        v1 = self.w0 * inputs + self.b0
        v2 = tf.math.sin(tf.matmul(inputs, self.w) + self.b)
        return tf.concat([v1, v2], axis=-1)

In [55]:
def transformer_encoder(x, head_size, num_heads, ff_dim, dropout=0):
    # Layer normalization
    x_norm = layers.LayerNormalization(epsilon=1e-6)(x)
    # Multi-head attention
    attention = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout)(x_norm, x_norm)
    # Skip connection
    x = layers.Add()([x, attention])
    x = layers.Dropout(dropout)(x)
    x_norm = layers.LayerNormalization(epsilon=1e-6)(x)
    # Feed-forward network
    ff = keras.Sequential([
        layers.Dense(ff_dim, activation="relu"),
        layers.Dense(x.shape[-1])
    ])
    x = layers.Add()([x_norm, ff(x_norm)])
    x = layers.Dropout(dropout)(x)
    return x


In [56]:

def build_model(input_shape, head_size, num_heads, num_classes, ff_dim, num_transformer_blocks, mlp_units, mlp_dropout, dropout):
    """
    Builds a transformer-based model with LSTM and dense layers.
    
    Parameters:
    - input_shape: Tuple (b, c), where b is sequence_length and c is number of features (e.g., 12 EMG channels).
    - head_size: Dimension of each attention head.
    - num_heads: Number of attention heads.
    - num_classes: Number of output classes.
    - ff_dim: Feed-forward dimension in transformer blocks (unused in this version).
    - num_transformer_blocks: Number of transformer blocks (set to 1 in original code).
    - mlp_units: List of units for MLP layers (e.g., [256]).
    - mlp_dropout: Dropout rate for MLP layers.
    - dropout: Dropout rate for attention and LSTM layers.
    """
    inputs = Input(shape=input_shape, name="input_layer")  # Shape: (b, c)
    
    # Initial Layer Normalization
    x = LayerNormalization(epsilon=1e-6, name="layer_normalization_4")(inputs)
    
    # Time2Vec Embedding
    x = Time2Vec(kernel_size=1)(x)  # Output shape: (b, c + 1)
    
    # First MultiHeadAttention Block
    x = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout, name="multi_head_attention_4"
    )(x, x)  # Output shape: (b, c + 1)
    x = Dropout(dropout, name="dropout_12")(x)
    
    # Residual connection: Project inputs to match Time2Vec output dimension
    inputs_projected = Dense(units=x.shape[-1], name="input_projection")(inputs)  # Shape: (b, c + 1)
    x = Add(name="tf.operators.add_6")([x, inputs_projected])
    x = LayerNormalization(epsilon=1e-6, name="layer_normalization_5")(x)
    
    # Second MultiHeadAttention Block
    res = x
    x = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout, name="multi_head_attention_5"
    )(x, x)  # Output shape: (b, c + 1)
    x = Dropout(dropout, name="dropout_13")(x)
    
    # Replace Multiply with Lambda layer for scaling (or no-op)
    x = Lambda(lambda x: x * 1.0, name="tf.math.multiply_2")(x)  # Output shape: (b, c + 1)
    x = Add(name="tf.operators.add_7")([x, res])
    x = LayerNormalization(epsilon=1e-6, name="layer_normalization_6")(x)
    
    # LSTM Layers
    x = LSTM(units=64, return_sequences=True, name="lstm_5")(x)  # Output shape: (b, 64)
    x = Dropout(dropout, name="dropout_14")(x)
    x = LSTM(units=64, return_sequences=True, name="lstm_6")(x)  # Output shape: (b, 64)
    
    # Residual connection: Project previous layer to match LSTM output dimension
    x_prev_projected = Dense(units=64, name="prev_projection")(res)  # Shape: (b, 64)
    x = Add(name="tf.operators.add_8")([x, x_prev_projected])
    
    # Global Average Pooling
    x = GlobalAveragePooling1D(name="global_average_pooling1d_2")(x)  # Output shape: (64,)
    
    # Dense Layers (Proxy for KolmogorovArnoldNetwork)
    x = Dense(units=mlp_units[0], activation="relu", name="kolmogorov_arnold_network_4")(x)
    x = Dropout(mlp_dropout, name="dropout_17")(x)
    x = Dense(units=mlp_units[0]//2, activation="relu", name="kolmogorov_arnold_network_5")(x)
    
    # Output Layer
    outputs = Dense(units=num_classes, activation="softmax", name="dense_7")(x)
    
    # Build Model
    model = Model(inputs=inputs, outputs=outputs, name="transformer_model")
    return model

In [57]:
def evaluate_model(trainX, trainy, testX, testy, sujet):
    verbose, epochs, batch_size = 1, 200, 64
    n_timesteps, d_model = trainX.shape[1], trainX.shape[2]
    n_outputs = trainy.shape[1]

    print("Train Data Shape:", trainX.shape)
    print("Test Data Shape:", testX.shape)

    model = build_model(
        input_shape=(n_timesteps, d_model),
        head_size=64,
        num_heads=16,
        num_classes=n_outputs,
        ff_dim=256,
        num_transformer_blocks=1,
        mlp_units=[256],
        mlp_dropout=0.6,
        dropout=0.5,
    )

    model.summary()
    model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

    start = time.time()
    model.fit(trainX, trainy, epochs=epochs, batch_size=batch_size, verbose=verbose, callbacks=[history])
    train_time = time.time() - start

    model.save(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\mlp=96, ff=96\models\transformer2' + str(sujet) + '_Transformer_model.h5')

    loss_train, accuracy_train = model.evaluate(trainX, trainy, batch_size=batch_size, verbose=1)
    start = time.time()
    loss_test, accuracy_test = model.evaluate(testX, testy, batch_size=batch_size, verbose=1)
    test_time = time.time() - start

    y_pred = np.argmax(model.predict(testX), axis=-1)
    testy_indices = [np.argmax(y) for y in testy]

    return loss_train, accuracy_train, loss_test, accuracy_test, y_pred, testy_indices, train_time, test_time


def summarize_results(scores, losses):
    m, s = mean(scores), std(scores)
    mL, sL = mean(losses), std(losses)
    print('\nAccuracy: %.5f (+/-%0.5f)' % (m, s))
    print('Loss: %.5f (+/-%0.5f)' % (mL, sL))


def run_my_experiment(sujet):
    data = loadmat(fr"D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\82_preprocessed_data\S" + str(sujet) + "_E1_A1_300_200_N.mat")

    train_data = data['train_data']
    train_labels = data['train_labels']
    test_data = data['test_data']
    test_labels = data['test_labels']

    scores, losses = [], []

    loss_train, score_train, loss_test, score_test, y_pred, testy, train_time, test_time = evaluate_model(
        train_data, train_labels, test_data, test_labels, sujet)

    print('>#%d: ' % (sujet))
    print('  train accuracy: %.5f' % (score_train))
    print('  train loss    : %.5f' % (loss_train))
    print('  test accuracy: %.5f' % (score_test))
    print('  test loss    : %.5f' % (loss_test))

    scores.append(score_test)
    losses.append(loss_test)
    summarize_results(scores, losses)

    return loss_train, score_train, loss_test, score_test, y_pred, testy, train_time, test_time


In [ ]:

# Main
# ==============================================================================

globel_perd = []
globel_class = []

globel_perd1 = []
globel_class1 = []

# Create a workbook and add a worksheet.
workbook = xlsxwriter.Workbook(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\mlp=96, ff=96\Trans_rslt.xlsx')
worksheet1 = workbook.add_worksheet('Subjects informations')

# Start from the first cell. Rows and columns are zero indexed.
row = 0

worksheet1.write(row, 0, 'Subject')
worksheet1.write(row, 1, 'Train_time')
worksheet1.write(row, 2, 'Test_time')
worksheet1.write(row, 3, 'Train_acc')
worksheet1.write(row, 4, 'Train_loss')
worksheet1.write(row, 5, 'Test_acc')
worksheet1.write(row, 6, 'Test_loss')
worksheet1.write(row, 7, 'MCC')

history = History()
for i in range(4, 5):
    loss_train, score_train, loss_test, score_test, y_pred, testy, train_time, test_time = run_my_experiment(i)

    globel_perd.append(y_pred)
    globel_class.append(testy)

    globel_perd1.extend(y_pred)
    globel_class1.extend(testy)

    mcc = matthews_corrcoef(testy, y_pred)
    mat = confusion_matrix(testy, y_pred)

    cfm_plot, ax = plot_confusion_matrix(mat, figsize=(10, 10), show_normed=True, show_absolute=False)
    cfm_plot.savefig(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\mlp=96, ff=96\S' + str(i) + 'trans_confusion_matrix.png')

    fig, axarr = plt.subplots(figsize=(12, 6), ncols=1)
    plot_renge = int(len(history.history['accuracy']) / i)
    global_renge = len(history.history['accuracy'])
    start_renge = global_renge - plot_renge
    axarr.plot(range(plot_renge), history.history['accuracy'][start_renge: global_renge], label='train score')
    axarr.plot(range(plot_renge), history.history['loss'][start_renge: global_renge], label='train loss')
    axarr.set_xlabel('Number of Epochs', fontsize=18)
    axarr.set_ylabel('Accuracy', fontsize=18)
    axarr.set_ylim([0, 2.5])
    plt.legend()
    plt.show()
    fig.savefig(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\mlp=96, ff=96\S' + str(i) + 'trans_graphe.png')

    # Sheet informations
    worksheet1.write(i, 0, 'Sujet ' + str(i))
    worksheet1.write(i, 1, train_time)
    worksheet1.write(i, 2, test_time)
    worksheet1.write(i, 3, score_train)
    worksheet1.write(i, 4, loss_train)
    worksheet1.write(i, 5, score_test)
    worksheet1.write(i, 6, loss_test)
    worksheet1.write(i, 7, mcc)

workbook.close()

# Save prediction
new_data = {'pred_labels': globel_perd, 'class_labels': globel_class}
savemat(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\mlp=96, ff=96\trans_global_predection.mat', new_data)

mat = confusion_matrix(globel_class1, globel_perd1)
cfm_plot, ax = plot_confusion_matrix(mat, figsize=(10, 10), show_normed=True, show_absolute=False)
cfm_plot.savefig(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\mlp=96, ff=96\trans_global_confusion_matrix.png')




Train Data Shape: (2641, 300, 12)
Test Data Shape: (1227, 300, 12)


Model: "transformer_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 12)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 300, 12)   │         24 │ input_layer[0][0] │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time2_vec_6         │ (None, 300, 13)   │         15 │ layer_normalizat… │
│ (Time2Vec)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 300, 13)   │     56,333 │ time2_vec_6[0][0… │
│ (MultiHeadAttentio… │                   │            │ time2_vec_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 300, 13)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_projection    │ (None, 300, 13)   │        169 │ input_layer[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tf.operators.add_6  │ (None, 300, 13)   │          0 │ dropout_12[0][0], │
│ (Add)               │                   │            │ input_projection… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 300, 13)   │         26 │ tf.operators.add… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 300, 13)   │     56,333 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 300, 13)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tf.math.multiply_2  │ (None, 300, 13)   │          0 │ dropout_13[0][0]  │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tf.operators.add_7  │ (None, 300, 13)   │          0 │ tf.math.multiply… │
│ (Add)               │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 300, 13)   │         26 │ tf.operators.add… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, 300, 64)   │     19,968 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 300, 64)   │          0 │ lstm_5[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ (None, 300, 64)   │     33,024 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prev_projection     │ (None, 300, 64)   │        896 │ layer_normalizat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 218,285 (852.68 KB)

 Trainable params: 218,285 (852.68 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
40/42 ━━━━━━━━━━━━━━━━━━━━ 16s 8s/step - accuracy: 0.0890 - loss: 2.6933